# Credit Card Approval Prediction - Epic 4: Model Building

This notebook covers the model training, evaluation, and selection stage. We train four classification models:
1. **Logistic Regression**
2. **Decision Tree**
3. **Random Forest**
4. **XGBoost**

We compare their performances and serialize the best-performing model as `best_model.pkl` for production use.

## 1. Import Core Libraries

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

import warnings
warnings.filterwarnings('ignore')
print("Model building libraries imported successfully.")

## 2. Load Processed Dataset

In [ ]:
data_path = os.path.join('..', '06_Data_Preprocessing', 'processed_dataset.csv')
df = pd.read_csv(data_path)

X = df.drop(columns=['y'])
y = df['y']
feature_names = X.columns.tolist()

print(f"Dataset Shape: {df.shape}")
print(f"Target distribution:\n{y.value_counts()}")

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 4. Helper Function for Model Evaluation

In [ ]:
results = {}
out_dir = os.path.join('..', '11_Outputs')
os.makedirs(out_dir, exist_ok=True)

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    start_time = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - start_time
    
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else np.zeros_like(y_pred)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob) if hasattr(model, "predict_proba") else 0.5
    
    results[name] = {
        'Model': model,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'AUC': auc,
        'Training_Time': train_time,
        'Prediction_Time': pred_time,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {acc:.4f}, AUC: {auc:.4f}")
    print(f"Training Time: {train_time:.4f}s, Inference Time: {pred_time:.4f}s\n")

## 5. Model Training

In [ ]:
# 5.1 Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
evaluate_model('Logistic Regression', lr, X_train, y_train, X_test, y_test)

# 5.2 Decision Tree
dt = DecisionTreeClassifier(random_state=42, max_depth=6)
evaluate_model('Decision Tree', dt, X_train, y_train, X_test, y_test)

# 5.3 Random Forest
rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
evaluate_model('Random Forest', rf, X_train, y_train, X_test, y_test)

# 5.4 XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss', max_depth=6, n_estimators=100)
evaluate_model('XGBoost', xgb, X_train, y_train, X_test, y_test)

## 6. Model Comparison

In [ ]:
comparison_list = []
for name, data in results.items():
    comparison_list.append({
        'Algorithm': name,
        'Accuracy': data['Accuracy'],
        'Precision': data['Precision'],
        'Recall': data['Recall'],
        'F1 Score': data['F1 Score'],
        'AUC': data['AUC'],
        'Training Time (s)': data['Training_Time'],
        'Prediction Time (s)': data['Prediction_Time']
    })

comparison_df = pd.DataFrame(comparison_list).sort_values(by='Accuracy', ascending=False)
print("--- Model Comparison Table ---")
print(comparison_df.to_string(index=False))

# Plot model comparison bar chart
plt.figure(figsize=(10, 5))
sns.barplot(x='Algorithm', y='Accuracy', data=comparison_df, palette='Blues_d')
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.ylim(0.7, 1.02)
plt.ylabel('Accuracy Score')
plt.savefig(os.path.join(out_dir, 'accuracy_comparison.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(out_dir, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Comparative Metrics Visualization (ROC Curve, Confusion Matrix, Feature Importance)

In [ ]:
# 7.1 ROC Curves Comparison
plt.figure(figsize=(10, 8))
for name, data in results.items():
    fpr, tpr, _ = roc_curve(y_test, data['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {data['AUC']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.legend(loc="lower right")
plt.savefig(os.path.join(out_dir, 'roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

# Get best model name
best_model_name = comparison_df.iloc[0]['Algorithm']
best_model_info = results[best_model_name]
print(f"Best Performing Model: {best_model_name}")

# 7.2 Confusion Matrix of the Best Model
cm = confusion_matrix(y_test, best_model_info['y_pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Approved', 'Rejected'], yticklabels=['Approved', 'Rejected'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.savefig(os.path.join(out_dir, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# 7.3 Feature Importance of Best Model (if applicable)
best_clf = best_model_info['Model']
if hasattr(best_clf, "feature_importances_"):
    importances = best_clf.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], palette='viridis')
    plt.title(f'Feature Importance - {best_model_name}', fontsize=14, fontweight='bold', color='#1E3A8A')
    plt.xlabel('Relative Importance')
    plt.ylabel('Feature')
    plt.savefig(os.path.join(out_dir, 'feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 8. Save Best Model

We save the best model file inside `07_Model_Building/` folder. We also copy the encoder and scaler here for self-containment.

In [ ]:
# Save model classifier
joblib.dump(best_clf, 'best_model.pkl')
print(f"Successfully saved {best_model_name} as best_model.pkl.")

# Copy scaler.pkl and encoder.pkl from Preprocessing directory if they exist there
import shutil
src_scaler = os.path.join('..', '06_Data_Preprocessing', 'scaler.pkl')
src_encoder = os.path.join('..', '06_Data_Preprocessing', 'encoder.pkl')

if os.path.exists(src_scaler):
    shutil.copy(src_scaler, 'scaler.pkl')
    print("Copied scaler.pkl to model building directory.")

if os.path.exists(src_encoder):
    shutil.copy(src_encoder, 'encoder.pkl')
    print("Copied encoder.pkl to model building directory.")